In [2]:
import os
import numpy as np
import mediapipe as mp
import cv2
import pickle

mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    min_detection_confidence=0.3,
    max_num_hands=1
)

DATA_PATH = './motion_data'
actions = os.listdir(DATA_PATH)

sequences = []
labels = []

def extract_features(hand_landmarks):
    x_, y_ = [], []

    for lm in hand_landmarks.landmark:
        x_.append(lm.x)
        y_.append(lm.y)

    features = []
    for lm in hand_landmarks.landmark:
        features.append(lm.x - min(x_))
        features.append(lm.y - min(y_))

    return features


for action_idx, action in enumerate(actions):
    for seq in os.listdir(os.path.join(DATA_PATH, action)):
        window = []

        for frame_num in range(30):
            img_path = os.path.join(DATA_PATH, action, seq, f"{frame_num}.jpg")
            img = cv2.imread(img_path)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            results = hands.process(img_rgb)

            if results.multi_hand_landmarks:
                hand_landmarks = results.multi_hand_landmarks[0]
                features = extract_features(hand_landmarks)
            else:
                features = [0]*42

            window.append(features)

        sequences.append(window)
        labels.append(action_idx)

X = np.array(sequences)
y = np.array(labels)

with open('motion_data.pickle', 'wb') as f:
    pickle.dump({'X': X, 'y': y, 'actions': actions}, f)

print("✅ Motion dataset created")
print("Shape:", X.shape)  # (samples, 30, 42)

✅ Motion dataset created
Shape: (300, 30, 42)
